# Globals

In [2]:
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time
import re
import os

# Output path folder creation
output = "/home/gaia/Desktop/MIPA/Output"
data = f"{output}/Data"
ana = f"{output}/Analysis"
os.makedirs(output, exist_ok=True)
os.makedirs(data, exist_ok=True)
os.makedirs(ana, exist_ok=True)

# Utils and Functions

## Support Functions - Parsing

In [ ]:
#Avvio il motore del broswer, per navigare nel sito + versione selenium x evitare sistemi anti-bot
def get_driver():
    options = uc.ChromeOptions()
    options.add_argument('--headless=new')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-gpu')
    options.add_argument('--window-size=1920,1080')
    try:
        driver = uc.Chrome(options=options, version_main=143)
    except:
        driver = uc.Chrome(options=options)
    return driver

#Estrazione dei dati che voglio es: settore, categoria ... 
def _extract_label(body_text, label):
    m = re.search(rf"{label}:\s*(.*)", body_text, re.IGNORECASE)
    return m.group(1).split('\n')[0].strip() if m else ""

#Estrazione del nome della procedura
def _extract_procedure_name(driver):
    nome_proc = "N/D"
    try:
        el = driver.find_element(By.XPATH, "//div[contains(@class, 'card-body')]//h3 | //div[contains(@class, 'card-body')]//h5 | //div[contains(@class, 'card-header')]//h5")
        nome_proc = el.text.strip()
    except:
        try:
            full_text = driver.find_element(By.TAG_NAME, "body").text
            parts = full_text.split("Dettaglio Procedura")[1].split("Settore:")[0].split('\n')
            valid_lines = [p.strip() for p in parts if p.strip() and "Scarica" not in p]
            if valid_lines: nome_proc = valid_lines[0]
        except:
            pass
    return nome_proc.replace("Scarica XLSX/CSV", "").strip()

#Trova TUTTI i box degli interventi nella pagina e restituisce una lista di dizionari (uno per ogni intervento)
def _parse_interventions(driver):
    interventions_list = []
    # Trova tutti i DIV che contengono "Tipologia di intervento" - XPATH cerca le card specifiche degli interventi, ignorando la card principale
    cards = driver.find_elements(By.XPATH, "//div[contains(@class, 'card') and .//*[contains(text(), 'Tipologia di intervento')]]")
    if not cards:
        # Se non trova restituisce un intervento vuoto (cella vuota) non perdere la riga
        return [{
            "Intervento": "", "Anno": "", "Descrizione Intervento": "",
            "Tipo Intervento": "", "Natura Intervento": "", "Riferimenti": ""
        }]

    for card in cards:
        try:
            card_text = card.text
            lines = card_text.split('\n')
            
            # Estraggo Anno e Titolo dal titolo della card o dalla prima riga
            try:
                title_el = card.find_element(By.TAG_NAME, "h5")
                title_full = title_el.text.strip()
            except:
                title_full = lines[0] if lines else ""

            anno = ""
            anno_m = re.search(r'\((\d{4})\)', title_full)
            if anno_m:
                anno = anno_m.group(1)
                title_clean = title_full.replace(f"({anno})", "").strip()
            else:
                title_clean = title_full

            # Estraggo dal box blu: Tipologia, Natura, Riferimenti
            def get_val_from_lines(keyword):
                for i, line in enumerate(lines):
                    if keyword.lower() in line.lower() and i + 1 < len(lines):
                        return lines[i+1].strip()
                return ""
            tipo = get_val_from_lines("Tipologia di intervento")
            natura = get_val_from_lines("Natura intervento")
            riferimenti = get_val_from_lines("Riferimenti")

            # Estraggo la descrizione: tutto ciò che c'è tra il titolo e l'inizio dei box blu
            desc = []
            start_capture = False
            for line in lines:
                clean_line = line.strip()
                if not clean_line: continue
                if clean_line in title_full:
                    start_capture = True
                    continue                
                if any(k in clean_line for k in ["Tipologia di intervento", "Natura intervento", "Riferimenti"]):
                    break               
                if start_capture:
                    desc.append(clean_line)
        
            if not desc and len(lines) > 4:
                 desc = [l for l in lines[1:] if "Tipologia" not in l and "Natura" not in l and "Riferimenti" not in l and len(l) > 20]

            interventions_list.append({
                "Intervento": title_clean,
                "Anno": anno,
                "Descrizione Intervento": " ".join(desc).strip(),
                "Tipo Intervento": tipo,
                "Natura Intervento": natura,
                "Riferimenti": riferimenti
            })
        except Exception as e:
            # Se un box fallisce, ne aggiunge uno vuoto per sicurezza
            interventions_list.append({"Intervento": "Errore parsing box", "Descrizione Intervento": str(e)})
    return interventions_list


#ORCHESTRATOR
def extract_page_rows(driver, current_id, url):
    """
    Restituisce una ista di singoli interventi per procedura (n righe Excel)
    """
    body_text = driver.find_element(By.TAG_NAME, "body").text

    base_data = {
        "ID": current_id,
        "Nome Procedura": _extract_procedure_name(driver),
        "Settore": _extract_label(body_text, "Settore"),
        "Categoria": _extract_label(body_text, "Categoria"),
        "Beneficiario": _extract_label(body_text, "Beneficiario"),
        "Tipo PA Responsabile": _extract_label(body_text, "Tipo PA Responsabile"),
        "URL": url
    }
    interventions = _parse_interventions(driver)
    rows = []
    for intervention in interventions:
        row = {**base_data, **intervention}
        rows.append(row)
    return rows



# Execution

In [ ]:
if __name__ == "__main__":
    # Range ID
    ID_START = 0
    ID_END = 700
    
    valid_data, error_data = run_scraper(ID_START, ID_END)

    # Salvataggio Dati Validi
    colonne_ordine = [
        "ID", "Nome Procedura", "Settore", "Categoria", "Beneficiario", 
        "Tipo PA Responsabile", #procedure 
        "Intervento", "Anno", "Descrizione Intervento", # interventi
        "Tipo Intervento", "Natura Intervento", "Riferimenti", 
        "URL"
    ]
    
    if valid_data:
        df = pd.DataFrame(valid_data)
        df = df.reindex(columns=colonne_ordine)
        df.to_excel(f"{data}/webscraping_long.xlsx", index=False)
        print(f"\nSono state salvate {len(valid_data)} righe (totale interventi) in webscraping_multi_row.xlsx")
    else:
        print("\nℹNessun dato valido.")

    #Salvo gli errori se ci sono in modo da poter controllare se gli errori sono dovuti ad un'assenza della pagina o a problemi di scraping
    if error_data:
        pd.DataFrame(error_data).to_excel(f"{data}/webscraping_errors.xlsx", index=False)

**!!!NB:**  gli ID--> numeri = [491, 493, 495, 496, 497, 498, 499, 501, 503, 504, 508, 511, 512, 513, 515, 516, 517, 518, 520, 521, 522, 523, 524, 526, 527, 528, 529, 530, 532, 534, 536, 538, 540, 542, 543, 544, 586]  HANNO COME TIPO INTERVENTO: "Tipologia di Intervento" invece di "Eliminazione/sostituzione di singole fasi o endoprocedimenti"

# Analysis 

In [ ]:
#Output path folder 
input_file = f"{data}/webscraping_long.xlsx"

#ANALISI VELOCE
def analizza_procedure(file_path):
    if not os.path.exists(file_path):
        print(f"Errore")
        return
    df = pd.read_excel(file_path)
    total_rows = len(df)                    # Quanti interventi
    total_procedures = df['ID'].nunique()   # Quante procedure (<<interventi)
    counts = df['ID'].value_counts()        # quante procedure con più di un intervento
    multi_intervention_procs = counts[counts > 1].count()
    single_intervention_procs = counts[counts == 1].count()

    print("="*40)
    print(f"Totale interventi:      {total_rows}")
    print(f"Totale procedure:    {total_procedures}")
    print("-" * 40)
    print(f"Procedure con un solo intervento: {single_intervention_procs}")
    print(f"Procedure con più di un intervento:  {multi_intervention_procs}")
    print("="*40)

analizza_procedure(input_file)

📂 Caricamento file: /home/gaia/Desktop/MIPA/Output/Data/webscraping_long.xlsx ...

📊 REPORT ANALISI PROCEDURE
🔹 Totale Righe (Interventi):      522
🔹 Totale PROCEDURE Reali (ID):    430
----------------------------------------
🔸 Procedure con 1 solo intervento: 380
🔸 Procedure con Multi-interventi:  50

📈 Distribuzione per Settore (Top 5):
Settore
Energia e fonti rinnovabili    166
Attività produttive             73
Cittadinanza digitale           48
Ambiente                        42
Infrastrutture digitali         24
Name: count, dtype: int64


In [4]:
import pandas as pd
import os

# ---------------------------------------------------------
# CONFIGURAZIONE PERCORSI
# ---------------------------------------------------------
try:
    input_file = os.path.join(data, "webscraping_long.xlsx")
    output_file = os.path.join(ana, "Report_Avanzato_Procedure.xlsx")
except NameError:
    print("⚠️ Variabili 'data' e 'ana' non definite. Uso cartella corrente.")
    input_file = "webscraping_long.xlsx"
    output_file = "Report_Avanzato_Procedure.xlsx"

# ---------------------------------------------------------
# FUNZIONE DI ANALISI
# ---------------------------------------------------------
def crea_report_excel(input_path, output_path):
    if not os.path.exists(input_path):
        print(f"❌ Errore: Il file {input_path} non esiste.")
        return

    print(f"📂 Lettura dati da: {input_path} ...")
    try:
        df = pd.read_excel(input_path)
    except Exception as e:
        print(f"❌ Errore nella lettura del file Excel: {e}")
        return

    # ---------------------------------------------------------
    # 1. PULIZIA E PREPARAZIONE DATI
    # ---------------------------------------------------------
    
    # Pulizia Anno
    if 'Anno' in df.columns:
        df['Anno_Clean'] = pd.to_numeric(df['Anno'], errors='coerce')
        df['Anno_Clean'] = df['Anno_Clean'].fillna(0).astype(int)
    
    # Creiamo df_unique per contare le PROCEDURE (1 ID = 1 Procedura)
    df_unique = df.drop_duplicates(subset=['ID']).copy()

    # ---------------------------------------------------------
    # 2. TABELLE DESCRITTIVE STANDARD
    # ---------------------------------------------------------

    # A. Riepilogo Generale (AGGIORNATO CON MIN/MAX)
    
    # Calcoli statistici sugli interventi per ID
    counts_per_id = df['ID'].value_counts()
    
    # Calcolo range anni (escludendo lo 0 che usiamo per i ND)
    anni_validi = df[df['Anno_Clean'] > 0]['Anno_Clean']
    min_anno = int(anni_validi.min()) if not anni_validi.empty else "ND"
    max_anno = int(anni_validi.max()) if not anni_validi.empty else "ND"

    summary_data = {
        'Metrica': [
            'Totale Righe (Interventi)', 
            'Totale Procedure Univoche (ID)', 
            'Media Interventi per Procedura',
            'Min Interventi in una singola Procedura',  # <--- NUOVO
            'Max Interventi in una singola Procedura',  # <--- NUOVO
            'Anno Inizio (Min)',                        # <--- NUOVO
            'Anno Fine (Max)',                          # <--- NUOVO
            'Settori distinti',
            'Categorie distinte'
        ],
        'Valore': [
            len(df),
            df['ID'].nunique(),
            round(counts_per_id.mean(), 2),
            counts_per_id.min(),                        # <--- CALCOLO MIN
            counts_per_id.max(),                        # <--- CALCOLO MAX
            min_anno,                                   # <--- ANNO MIN
            max_anno,                                   # <--- ANNO MAX
            df_unique['Settore'].nunique() if 'Settore' in df else 0,
            df_unique['Categoria'].nunique() if 'Categoria' in df else 0
        ]
    }
    df_summary = pd.DataFrame(summary_data)

    # B. Analisi Tipo e Natura Intervento
    if 'Tipo Intervento' in df.columns:
        df_tipo_int = df['Tipo Intervento'].value_counts().reset_index()
        df_tipo_int.columns = ['Tipo Intervento', 'Numero Interventi']
    else:
        df_tipo_int = pd.DataFrame()

    if 'Natura Intervento' in df.columns:
        df_natura_int = df['Natura Intervento'].value_counts().reset_index()
        df_natura_int.columns = ['Natura Intervento', 'Numero Interventi']
    else:
        df_natura_int = pd.DataFrame()

    # C. Analisi Anagrafiche
    if 'Settore' in df_unique.columns:
        df_settore = df_unique['Settore'].value_counts().reset_index()
        df_settore.columns = ['Settore', 'Numero Procedure']
        df_settore['%'] = (df_settore['Numero Procedure'] / len(df_unique) * 100).round(2)
    else: df_settore = pd.DataFrame()

    if 'Categoria' in df_unique.columns:
        df_cat = df_unique['Categoria'].value_counts().reset_index()
        df_cat.columns = ['Categoria', 'Numero Procedure']
    else: df_cat = pd.DataFrame()

    if 'Tipo PA Responsabile' in df_unique.columns:
        df_pa = df_unique['Tipo PA Responsabile'].value_counts().reset_index()
        df_pa.columns = ['Tipo PA', 'Numero Procedure']
    else: df_pa = pd.DataFrame()

    # ---------------------------------------------------------
    # 3. TABELLE PIVOT
    # ---------------------------------------------------------
    pivot_combinations = {
        "Pivot_Anno_Settore":    ("Anno_Clean", "Settore"),
        "Pivot_Anno_Categoria":  ("Anno_Clean", "Categoria"),
        "Pivot_PA_Settore":      ("Tipo PA Responsabile", "Settore"),
        "Pivot_Settore_Cat":     ("Settore", "Categoria"),
        "Pivot_Anno_TipoPA":     ("Anno_Clean", "Tipo PA Responsabile"),
        "Pivot_Benefic_Settore": ("Beneficiario", "Settore")
    }

    pivots_generated = {}

    for sheet_name, (col_index, col_columns) in pivot_combinations.items():
        if col_index in df_unique.columns and col_columns in df_unique.columns:
            pivot = pd.crosstab(
                df_unique[col_index], 
                df_unique[col_columns], 
                margins=True, 
                margins_name="TOTALE"
            )
            if col_index == "Beneficiario" and len(pivot) > 50:
                pivot = pivot.sort_values("TOTALE", ascending=False).head(50)
            pivots_generated[sheet_name] = pivot
        else:
            print(f"⚠️ Impossibile creare pivot {sheet_name}: colonne mancanti.")

    # ---------------------------------------------------------
    # 4. SALVATAGGIO SU EXCEL
    # ---------------------------------------------------------
    print(f"💾 Salvataggio report in: {output_path} ...")
    
    try:
        with pd.ExcelWriter(output_path, engine='xlsxwriter') as writer:
            
            # Fogli Descrittivi
            df_summary.to_excel(writer, sheet_name='Riepilogo', index=False)
            
            df_tipo_int.to_excel(writer, sheet_name='Analisi Interventi', startrow=0, index=False)
            df_natura_int.to_excel(writer, sheet_name='Analisi Interventi', startrow=0, startcol=4, index=False)
            
            df_settore.to_excel(writer, sheet_name='Anagrafiche', startrow=0, index=False)
            df_cat.to_excel(writer, sheet_name='Anagrafiche', startrow=0, startcol=4, index=False)
            df_pa.to_excel(writer, sheet_name='Anagrafiche', startrow=0, startcol=8, index=False)

            # Fogli Pivot
            for sheet_name, pivot_df in pivots_generated.items():
                pivot_df.to_excel(writer, sheet_name=sheet_name)

            # Formattazione
            workbook = writer.book
            
            # Formattazione Riepilogo
            ws_summ = writer.sheets['Riepilogo']
            ws_summ.set_column(0, 0, 35) # Allarga colonna Metrica
            ws_summ.set_column(1, 1, 15) # Allarga colonna Valore

            for sheet_name in writer.sheets:
                if sheet_name != 'Riepilogo':
                    writer.sheets[sheet_name].set_column(0, 10, 20)

        print("✅ Report avanzato creato con successo!")
        
    except Exception as e:
        print(f"❌ Errore durante il salvataggio: {e}")

if __name__ == "__main__":
    if 'ana' in locals() and not os.path.exists(ana):
        try: os.makedirs(ana)
        except: pass
    crea_report_excel(input_file, output_file)

📂 Lettura dati da: /home/gaia/Desktop/MIPA/Output/Data/webscraping_long.xlsx ...
💾 Salvataggio report in: /home/gaia/Desktop/MIPA/Output/Analysis/Report_Avanzato_Procedure.xlsx ...
✅ Report avanzato creato con successo!


In [10]:
import pandas as pd
import os

# ---------------------------------------------------------
# CONFIGURAZIONE PERCORSI
# ---------------------------------------------------------
try:
    input_file = os.path.join(data, "webscraping_long.xlsx")
    output_report = os.path.join(ana, "Report_Procedure_vs_Interventi.xlsx")
except NameError:
    print("⚠️ Variabili 'data' e 'ana' non definite. Uso cartella corrente.")
    input_file = "webscraping_long.xlsx"
    output_report = "Report_Procedure_vs_Interventi.xlsx"

# ---------------------------------------------------------
# FUNZIONE DI ANALISI DOPPIA
# ---------------------------------------------------------
def genera_report_doppio(input_path, output_path):
    if not os.path.exists(input_path):
        print(f"❌ Errore: File {input_path} non trovato.")
        return

    print(f"📂 Lettura dati: {input_path} ...")
    try:
        df = pd.read_excel(input_path)
    except Exception as e:
        print(f"❌ Errore lettura Excel: {e}")
        return

    # =========================================================
    # CONFIGURAZIONE VARIABILI DA ANALIZZARE
    # =========================================================
    
    # 1. Variabili per Tabella PROCEDURE (richiedono ID unico)
    vars_procedure = ['Settore', 'Beneficiario', 'Categoria', 'Tipo PA Responsabile']
    
    # 2. Variabili per Tabella INTERVENTI (richiedono tutte le righe)
    vars_interventi = ['Anno', 'Tipo Intervento', 'Natura Intervento']

    # Pulizia preliminare Anno (per evitare 2020.0)
    if 'Anno' in df.columns:
        df['Anno'] = pd.to_numeric(df['Anno'], errors='coerce').fillna(0).astype(int)
        # Sostituiamo 0 con "Dato Mancante" per il report
        df['Anno'] = df['Anno'].replace(0, 'Dato Mancante')

    # =========================================================
    # ELABORAZIONE TABELLA 1: PROCEDURE
    # =========================================================
    print("⚙️ Elaborazione Tabella 1: PROCEDURE (Settore, Beneficiario, Categoria, PA)...")
    
    # Isoliamo gli ID univoci
    df_unique = df.drop_duplicates(subset=['ID']).copy()
    tot_procedure = len(df_unique)
    print(f"   Base dati: {tot_procedure} procedure uniche.")
    
    dati_procedure = []
    
    for col in vars_procedure:
        # Verifichiamo se la colonna esiste nel file
        if col not in df_unique.columns:
            print(f"   ⚠️ Attenzione: Colonna '{col}' non trovata nel file.")
            continue
            
        conteggi = df_unique[col].value_counts(dropna=False)
        
        for valore, n in conteggi.items():
            pct = n / tot_procedure
            val_str = "Dato Mancante (NULL)" if pd.isna(valore) or valore == "" else str(valore)
            
            dati_procedure.append({
                "Variabile": col,
                "Categoria": val_str,
                "N (Procedure)": n,
                "%": pct
            })

    df_tab1 = pd.DataFrame(dati_procedure)

    # =========================================================
    # ELABORAZIONE TABELLA 2: INTERVENTI
    # =========================================================
    print("⚙️ Elaborazione Tabella 2: INTERVENTI (Anno, Tipo, Natura)...")
    
    # Usiamo il DF completo (tutte le righe)
    tot_interventi = len(df)
    print(f"   Base dati: {tot_interventi} righe totali.")
    
    dati_interventi = []
    
    for col in vars_interventi:
        if col not in df.columns:
            print(f"   ⚠️ Attenzione: Colonna '{col}' non trovata nel file.")
            continue
            
        conteggi = df[col].value_counts(dropna=False)
        
        for valore, n in conteggi.items():
            pct = n / tot_interventi
            val_str = "Dato Mancante (NULL)" if pd.isna(valore) or valore == "" else str(valore)
            
            dati_interventi.append({
                "Variabile": col,
                "Categoria": val_str,
                "N (Interventi)": n,
                "%": pct
            })

    df_tab2 = pd.DataFrame(dati_interventi)

    # =========================================================
    # SALVATAGGIO SU EXCEL
    # =========================================================
    print(f"💾 Salvataggio in: {output_path} ...")
    try:
        with pd.ExcelWriter(output_path, engine='xlsxwriter') as writer:
            
            # --- FOGLIO 1: PROCEDURE ---
            df_tab1.to_excel(writer, sheet_name='Analisi PROCEDURE', index=False)
            
            # --- FOGLIO 2: INTERVENTI ---
            df_tab2.to_excel(writer, sheet_name='Analisi INTERVENTI', index=False)
            
            # --- FORMATTAZIONE COMUNE ---
            workbook = writer.book
            pct_fmt = workbook.add_format({'num_format': '0.00%'})
            header_fmt = workbook.add_format({'bold': True, 'fg_color': '#DDEBF7', 'border': 1})
            bold_fmt = workbook.add_format({'bold': True, 'valign': 'top'})
            
            for sheet_name in ['Analisi PROCEDURE', 'Analisi INTERVENTI']:
                worksheet = writer.sheets[sheet_name]
                
                # Larghezza colonne
                worksheet.set_column('A:A', 25, bold_fmt)   # Variabile
                worksheet.set_column('B:B', 50)             # Categoria
                worksheet.set_column('C:C', 15)             # N
                worksheet.set_column('D:D', 12, pct_fmt)    # %
                
                # Intestazione colorata
                df_curr = df_tab1 if sheet_name == 'Analisi PROCEDURE' else df_tab2
                for col_num, value in enumerate(df_curr.columns.values):
                    worksheet.write(0, col_num, value, header_fmt)

                # Blocca riquadri e filtri
                worksheet.freeze_panes(1, 0)
                worksheet.autofilter(0, 0, len(df_curr), 3)

        print("✅ Report creato con successo!")
        print("   Foglio 1: Basato su ID univoci (Procedure).")
        print("   Foglio 2: Basato su righe totali (Interventi).")

    except Exception as e:
        print(f"❌ Errore salvataggio: {e}")

# ---------------------------------------------------------
# MAIN
# ---------------------------------------------------------
if __name__ == "__main__":
    if 'ana' in locals() and not os.path.exists(ana):
        try: os.makedirs(ana)
        except: pass
        
    genera_report_doppio(input_file, output_report)

📂 Lettura dati: /home/gaia/Desktop/MIPA/Output/Data/webscraping_long.xlsx ...
⚙️ Elaborazione Tabella 1: PROCEDURE (Settore, Beneficiario, Categoria, PA)...
   Base dati: 430 procedure uniche.
⚙️ Elaborazione Tabella 2: INTERVENTI (Anno, Tipo, Natura)...
   Base dati: 522 righe totali.
💾 Salvataggio in: /home/gaia/Desktop/MIPA/Output/Analysis/Report_Procedure_vs_Interventi.xlsx ...
✅ Report creato con successo!
   Foglio 1: Basato su ID univoci (Procedure).
   Foglio 2: Basato su righe totali (Interventi).
